# AutoMark – train the handwriting model on the FULL SSL-HWD-Words dataset (Kaggle)

Trains our reader (**CNN 5 layers + BiLSTM 2 layers + CTC**, same network as `grader/htr_model.py`) on
**every usable labelled word** of SSL-HWD-Words (about 2 million words), not a sample.
Only this Hugging Face dataset is used.

- Starts from the **round-2 model** (best so far) if you added it as input; otherwise from the app weights on GitHub.
- Validation and test words are the **same writers as rounds 1 and 2** (seed 42), so results compare directly.
  Those writers are never used for training.
- Images are read straight from the zip while training (2 million images do not fit in memory).
- One epoch is about 2 million words: **roughly 30–45 minutes**. Kaggle stops a run after 12 hours,
  so the notebook stops itself after `TIME_LIMIT_HOURS` and still saves everything.
- After every epoch: best model saved, gradients printed, all numbers written to separate txt files.
- If a run is cut short, add `results_full.zip`'s `last_checkpoint.pt` as input and run again: it continues.

Everything is saved in `/kaggle/working/results_full/` and zipped to `/kaggle/working/results_full.zip`.

| File | What it holds |
|---|---|
| `train_loss.txt`, `val_loss.txt` | training / validation CTC loss per epoch |
| `val_cer.txt`, `val_word_accuracy.txt` | validation character error rate % and words fully correct % per epoch |
| `grad_norm.txt`, `grad_norm_per_layer.txt` | gradient norms per epoch (total and per layer) |
| `learning_rate.txt`, `epoch_time.txt`, `history.csv` | learning rate, time, and everything in one table |
| `*.png` | graphs for the report |
| `test_results.txt`, `sample_predictions.txt` | final test: start model vs trained model |
| `htr_weights_best.npz` | **best model in the app's format** → `automark/model/htr_weights.npz` |
| `best_model.pt`, `last_checkpoint.pt` | PyTorch copies (checkpoint is for continuing) |


## 1. Settings

In [ ]:
# ---------------- data
MAX_TRAIN_WORDS = None      # None = ALL labelled words (about 2 million). A number = use only that many.
N_VAL_WORDS     = 10_000    # same as rounds 1 and 2 (do not change, or results stop being comparable)
N_TEST_WORDS    = 10_000
MAX_PER_WRITER  = 300       # only used to pick the validation/test words, exactly as in round 1
MAX_LABEL_LEN   = 24        # longer words are skipped (the model has 32 time steps)

# ---------------- training
EPOCHS          = 12        # upper limit; the time limit below usually decides
TIME_LIMIT_HOURS = 11.0     # stop starting new epochs after this (Kaggle kills runs at 12 h)
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-4
PATIENCE        = 4         # stop when validation CER has not improved for this many epochs
NUM_WORKERS     = 4         # Kaggle gives 4 CPU cores
SEED            = 42        # keep 42: picks the same validation/test writers as rounds 1 and 2

# ---------------- paths
HF_REPO   = "Shree10/SSL-HWD-Words"
HF_FILE   = "SSL-HWD-Words(Labeled).zip"
DATA_DIR  = "/tmp/sslhwd"
GITHUB    = "https://github.com/demonayush11/Automark.git"
OUT       = "/kaggle/working/results_full"

import os, time
RUN_START = time.time()
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/config.txt", "w") as f:
    for k, v in dict(globals()).items():
        if k.isupper() and k != "RUN_START":
            f.write(f"{k} = {v}\n")
print("Settings saved to", f"{OUT}/config.txt")

## 2. GPU

In [ ]:
!pip -q install -U huggingface_hub hf_transfer
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU - training will be very slow")


## 3. Model files: which weights to start from

In [ ]:
import glob, subprocess
if not os.path.exists("/kaggle/working/automark"):
    subprocess.run(["git", "clone", "-q", GITHUB, "/kaggle/working/automark"])

def first(patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return None

CHARS_FILE = first(["/kaggle/working/automark/**/charList.txt", "/kaggle/input/**/charList.txt"])
assert CHARS_FILE, "charList.txt not found: GitHub clone failed and no input has it"
CHARS = open(CHARS_FILE, encoding="utf-8").read()

# start model: round-2 model if added as input, else the app's current weights
START_WEIGHTS = first(["/kaggle/input/**/*round2*/**/htr_weights_best.npz",
                       "/kaggle/input/**/htr_weights_best.npz",
                       "/kaggle/working/results_round2/htr_weights_best.npz",
                       "/kaggle/input/**/htr_weights.npz",
                       "/kaggle/working/automark/model/htr_weights.npz"])
assert START_WEIGHTS, "no htr_weights file found"
# the untouched IAM model, only for the comparison table (optional)
ORIGINAL_WEIGHTS = first(["/kaggle/input/**/htr_weights_original.npz",
                          "/kaggle/working/automark/model/htr_weights_original.npz"])
RESUME_FILE = first(["/kaggle/input/**/last_checkpoint.pt"])
print("Start weights   :", START_WEIGHTS)
print("Original weights:", ORIGINAL_WEIGHTS or "not found (comparison row skipped)")
print("Resume from     :", RESUME_FILE or "nothing - fresh run")
print("Characters the model knows:", len(CHARS))

## 4. Download the dataset (14.5 GB, a few minutes)

In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
ZIP_PATH = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, repo_type="dataset",
                           local_dir=DATA_DIR, token=token)
print("Downloaded:", ZIP_PATH, round(os.path.getsize(ZIP_PATH) / 1e9, 2), "GB")
!df -h /tmp | tail -1


## 5. Look inside the zip

In [ ]:
import zipfile, posixpath, io, collections
import pandas as pd

zf = zipfile.ZipFile(ZIP_PATH)
NAMES = [n for n in zf.namelist() if not n.startswith("__MACOSX") and not n.endswith("/")]
CSVS = sorted(n for n in NAMES if n.lower().endswith(".csv"))
IMAGES = [n for n in NAMES if n.lower().endswith((".png", ".jpg", ".jpeg"))]
print(f"{len(NAMES):,} files: {len(IMAGES):,} images, {len(CSVS):,} label files")
top = collections.Counter("/".join(n.split("/")[:2]) for n in NAMES)
print("Top folders:", list(top.most_common(8)))
print("Example label files:", CSVS[:3])
print(pd.read_csv(io.BytesIO(zf.read(CSVS[0]))).head())


## 6. Choose the words
Validation and test: picked exactly as in round 1 (same seed, same writers). Training: **every** usable
labelled word of all other writers. Only names are listed here; images are read during training.

In [ ]:
import random, cv2, numpy as np
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED)
CHARSET = set(CHARS)
NAME_SET = set(NAMES)
_by_base = None
def resolve(folder, fname):
    """Path of an image inside the zip, from the label file's folder and the file name."""
    global _by_base
    p = posixpath.normpath(posixpath.join(folder, fname))
    if p in NAME_SET:
        return p
    if _by_base is None:
        _by_base = {}
        for n in IMAGES:
            _by_base.setdefault(posixpath.basename(n), n)
    return _by_base.get(posixpath.basename(fname))

def read_labels(csv_path):
    df = pd.read_csv(io.BytesIO(zf.read(csv_path)), dtype=str, keep_default_na=False)
    cols = {c.lower().strip(): c for c in df.columns}
    fcol = cols.get("image_filename", df.columns[0])
    tcol = cols.get("transcription", df.columns[1])
    rows = []
    for f, t in zip(df[fcol], df[tcol]):
        t = str(t).strip()
        if 1 <= len(t) <= MAX_LABEL_LEN and all(c in CHARSET for c in t):
            rows.append((f, t))
    return rows

def decode_gray(data):
    """Bytes of an image -> grayscale, dark text on light background, height <= 64, width <= 512."""
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_GRAYSCALE)
    if img is None or img.size == 0:
        return None
    if img.mean() < 110:
        img = 255 - img
    h, w = img.shape
    s = min(1.0, 64 / h, 512 / w)
    if s < 1:
        img = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    return img

load_gray = lambda path: decode_gray(zf.read(path))

# --- validation and test: identical procedure to round 1 (so the same writers and words)
csv_order = CSVS[:]
random.shuffle(csv_order)
splits = {"test": [], "val": []}
goal = {"test": N_TEST_WORDS, "val": N_VAL_WORDS}
writers = {"test": [], "val": []}
for csv_path in tqdm(csv_order, desc="validation/test writers"):
    part = next((k for k in ("test", "val") if len(splits[k]) < goal[k]), None)
    if part is None:
        break
    folder = posixpath.dirname(csv_path)
    rows = read_labels(csv_path)
    random.shuffle(rows)
    took = 0
    for fname, text in rows:
        if took >= MAX_PER_WRITER or len(splits[part]) >= goal[part]:
            break
        p = resolve(folder, fname)
        img = load_gray(p) if p else None
        if img is None:
            continue
        splits[part].append((img, text)); took += 1
    if took:
        writers[part].append(folder)
held_out = set(writers["test"]) | set(writers["val"])

# --- training: every usable word of every other writer (names only)
train_names, train_writers, missing = [], set(), 0
for csv_path in tqdm(CSVS, desc="training label files"):
    folder = posixpath.dirname(csv_path)
    if folder in held_out:
        continue
    for fname, text in read_labels(csv_path):
        p = resolve(folder, fname)
        if p is None:
            missing += 1
            continue
        train_names.append((p, text)); train_writers.add(folder)
random.Random(SEED).shuffle(train_names)
if MAX_TRAIN_WORDS:
    train_names = train_names[:MAX_TRAIN_WORDS]

info = [f"Dataset: {HF_REPO} / {HF_FILE}",
        f"Label files in zip: {len(CSVS)}; images in zip: {len(IMAGES):,}",
        f"train: {len(train_names):,} words from {len(train_writers)} writers (ALL usable labelled words of these writers)",
        f"val  : {len(splits['val']):,} words from {len(writers['val'])} writers (same as rounds 1 and 2)",
        f"test : {len(splits['test']):,} words from {len(writers['test'])} writers (same as rounds 1 and 2)",
        f"Label rows whose image was not found: {missing}",
        f"Labels kept only if 1-{MAX_LABEL_LEN} characters, all among the model's {len(CHARS)} characters",
        "Writers never overlap between train, val and test."]
print("\n".join(info))
open(f"{OUT}/dataset_info.txt", "w").write("\n".join(info) + "\n")
assert train_names and splits["val"] and splits["test"], "Not enough data found - check cell 5 output"

import matplotlib.pyplot as plt
fig, axes = plt.subplots(6, 4, figsize=(12, 9))
for ax, (p, text) in zip(axes.flat, random.sample(train_names, 24)):
    ax.imshow(load_gray(p), cmap="gray"); ax.set_title(text, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/sample_training_words.png", dpi=110); plt.show()

## 7. The model (same network as the app)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
IMG_W, IMG_H, T_STEPS, HIDDEN = 128, 32, 32, 256

class HTR(nn.Module):
    """5 x (conv -> batch-norm -> ReLU -> max-pool) -> LSTM read forwards and backwards with the
    SAME weights -> linear layer to 78 classes (77 characters + CTC blank). Same as grader/htr_model.py."""
    def __init__(self, n_classes):
        super().__init__()
        chans, kernels = [1, 32, 64, 128, 128, 256], [5, 5, 3, 3, 3]
        self.pools = [(2, 2), (2, 2), (1, 2), (1, 2), (1, 2)]
        self.convs = nn.ModuleList(nn.Conv2d(chans[i], chans[i + 1], kernels[i], padding=kernels[i] // 2, bias=False)
                                   for i in range(5))
        self.bns = nn.ModuleList(nn.BatchNorm2d(chans[i + 1], eps=1e-3) for i in range(5))
        self.lstm = nn.LSTM(256, HIDDEN, num_layers=2, batch_first=True)
        self.out = nn.Linear(2 * HIDDEN, n_classes, bias=False)

    def forward(self, x):                              # x: (B, 128, 32)
        x = x.unsqueeze(1)
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = F.max_pool2d(F.relu(bn(conv(x))), pool)
        seq = x.squeeze(3).permute(0, 2, 1)            # (B, 32, 256)
        fw, _ = self.lstm(seq)
        bw, _ = self.lstm(torch.flip(seq, [1]))
        return self.out(torch.cat([fw, torch.flip(bw, [1])], dim=2))   # (B, 32, 78)

LSTM_KEY = "bidirectional_rnn__fw__multi_rnn_cell__cell_{}__lstm_cell__{}"

def load_npz(model, path):
    """App weights (TensorFlow layout) -> PyTorch."""
    w = np.load(path)
    t = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=torch.float32)
    with torch.no_grad():
        for i in range(5):
            model.convs[i].weight.copy_(t(w["Variable" + ("" if i == 0 else f"_{i}")].transpose(3, 2, 0, 1)))
            bn = "batch_normalization" + ("" if i == 0 else f"_{i}")
            model.bns[i].weight.copy_(t(w[bn + "__gamma"])); model.bns[i].bias.copy_(t(w[bn + "__beta"]))
            model.bns[i].running_mean.copy_(t(w[bn + "__moving_mean"]))
            model.bns[i].running_var.copy_(t(w[bn + "__moving_variance"]))
        for layer in range(2):
            kernel, bias = w[LSTM_KEY.format(layer, "kernel")], w[LSTM_KEY.format(layer, "bias")]
            n_in = kernel.shape[0] - HIDDEN
            gi, gj, gf, go = np.split(kernel, 4, axis=1); bi, bj, bf, bo = np.split(bias, 4)
            k = np.concatenate([gi, gf, gj, go], axis=1)
            getattr(model.lstm, f"weight_ih_l{layer}").copy_(t(k[:n_in].T))
            getattr(model.lstm, f"weight_hh_l{layer}").copy_(t(k[n_in:].T))
            getattr(model.lstm, f"bias_ih_l{layer}").copy_(t(np.concatenate([bi, bf + 1.0, bj, bo])))
            getattr(model.lstm, f"bias_hh_l{layer}").zero_()
        model.out.weight.copy_(t(w["Variable_5"][0, 0].T))

def save_npz(model, path):
    """PyTorch -> app weights file (put it in automark/model/htr_weights.npz)."""
    a = lambda p: p.detach().cpu().numpy().astype(np.float32)
    d = {}
    for i in range(5):
        d["Variable" + ("" if i == 0 else f"_{i}")] = a(model.convs[i].weight).transpose(2, 3, 1, 0)
        bn = "batch_normalization" + ("" if i == 0 else f"_{i}")
        d[bn + "__gamma"], d[bn + "__beta"] = a(model.bns[i].weight), a(model.bns[i].bias)
        d[bn + "__moving_mean"], d[bn + "__moving_variance"] = a(model.bns[i].running_mean), a(model.bns[i].running_var)
    for layer in range(2):
        w_ih, w_hh = a(getattr(model.lstm, f"weight_ih_l{layer}")), a(getattr(model.lstm, f"weight_hh_l{layer}"))
        b = a(getattr(model.lstm, f"bias_ih_l{layer}")) + a(getattr(model.lstm, f"bias_hh_l{layer}"))
        gi, gf, gg, go = np.split(np.concatenate([w_ih.T, w_hh.T], axis=0), 4, axis=1)
        bi, bf, bg, bo = np.split(b, 4)
        d[LSTM_KEY.format(layer, "kernel")] = np.concatenate([gi, gg, gf, go], axis=1)
        d[LSTM_KEY.format(layer, "bias")] = np.concatenate([bi, bg, bf - 1.0, bo])
    d["Variable_5"] = a(model.out.weight).T[None, None]
    np.savez_compressed(path, **d)

def new_model(path):
    m = HTR(len(CHARS) + 1); load_npz(m, path); return m.to(DEVICE)

model = new_model(START_WEIGHTS)
print("Started from:", START_WEIGHTS)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Batches (training images are read from the zip by 4 helper processes)

In [ ]:
from torch.utils.data import Dataset, DataLoader
CHAR_INDEX = {c: i for i, c in enumerate(CHARS)}
BLANK = len(CHARS)

def preprocess(gray):
    """Exactly as the app (grader/htr_model.py): fit into 128x32 on white, transpose, normalise."""
    h, w = gray.shape
    f = max(w / IMG_W, h / IMG_H)
    nw, nh = max(min(IMG_W, int(w / f)), 1), max(min(IMG_H, int(h / f)), 1)
    canvas = np.ones((IMG_H, IMG_W)) * 255
    canvas[:nh, :nw] = cv2.resize(gray, (nw, nh))
    img = cv2.transpose(canvas)
    m, s = img.mean(), img.std()
    return ((img - m) / s if s > 0 else img - m).astype(np.float32)

def augment(gray, rng):
    """Small random changes so the model learns the writing, not the exact pixels."""
    h, w = gray.shape
    gray = cv2.resize(gray, (max(1, int(w * rng.uniform(0.8, 1.2))), h))      # wider / narrower
    h, w = gray.shape
    m = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-3, 3), 1)        # small rotation
    m[0, 1] += rng.uniform(-0.25, 0.25)                                       # slant
    gray = cv2.warpAffine(gray, m, (w, h), borderValue=255)
    r = rng.random()
    if r < 0.25:   gray = cv2.erode(gray, np.ones((2, 2), np.uint8))          # thicker ink
    elif r < 0.5:  gray = cv2.dilate(gray, np.ones((2, 2), np.uint8))         # thinner ink
    if rng.random() < 0.3:
        gray = cv2.GaussianBlur(gray, (3, 3), 0)                              # slight blur
    return gray

class Words(Dataset):
    """Validation/test: images already in memory."""
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        img, text = self.items[i]
        return torch.from_numpy(preprocess(img)), [CHAR_INDEX[c] for c in text], text

class ZipWords(Dataset):
    """Training: reads each image from the zip when needed. Every helper process opens its own zip handle."""
    def __init__(self, names, zip_path):
        self.names, self.zip_path, self.zf, self.pid = names, zip_path, None, None
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        if self.pid != os.getpid():
            self.zf, self.pid = zipfile.ZipFile(self.zip_path), os.getpid()
        rng = np.random.default_rng()
        for _ in range(10):                          # an unreadable image -> use another random one
            p, text = self.names[i]
            img = decode_gray(self.zf.read(p))
            if img is not None:
                break
            i = int(rng.integers(len(self.names)))
        img = augment(img, rng)
        return torch.from_numpy(preprocess(img)), [CHAR_INDEX[c] for c in text], text

def collate(batch):
    x = torch.stack([b[0] for b in batch])
    targets = torch.tensor([c for b in batch for c in b[1]], dtype=torch.long)
    lengths = torch.tensor([len(b[1]) for b in batch], dtype=torch.long)
    return x, targets, lengths, [b[2] for b in batch]


train_loader = DataLoader(ZipWords(train_names, ZIP_PATH), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=(DEVICE == "cuda"),
                          persistent_workers=True, prefetch_factor=4)
val_loader = DataLoader(Words(splits["val"]), batch_size=BATCH_SIZE, collate_fn=collate)
test_loader = DataLoader(Words(splits["test"]), batch_size=BATCH_SIZE, collate_fn=collate)
print("Training batches per epoch:", f"{len(train_loader):,}")

## 9. Loss and measuring

In [ ]:
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)

def decode(logits):
    out = []
    for seq in logits.argmax(2).cpu().numpy():
        txt, prev = [], None
        for k in seq:
            if k != prev and k != BLANK:
                txt.append(CHARS[k])
            prev = k
        out.append("".join(txt))
    return out

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

@torch.no_grad()
def evaluate(model, data_loader, keep_examples=0):
    """Returns loss, CER %, word accuracy %, and some (truth, prediction) examples."""
    model.eval()
    loss_sum = n = errs = chars = right = 0
    examples = []
    for x, targets, lengths, texts in data_loader:
        logits = model(x.to(DEVICE))
        logp = F.log_softmax(logits, 2).permute(1, 0, 2)
        loss = ctc_loss(logp, targets, torch.full((len(texts),), T_STEPS, dtype=torch.long), lengths)
        loss_sum += loss.item() * len(texts); n += len(texts)
        for pred, truth in zip(decode(logits), texts):
            errs += edit_distance(pred, truth); chars += len(truth); right += pred == truth
            if len(examples) < keep_examples:
                examples.append((truth, pred))
    return loss_sum / n, 100 * errs / chars, 100 * right / n, examples


## 10. Score of the start model (epoch 0)

In [ ]:
base_val = evaluate(model, val_loader)
base_test = evaluate(model, test_loader, keep_examples=40)
print(f"Start model | val : CER {base_val[1]:.2f}%  words correct {base_val[2]:.2f}%")
print(f"Start model | test: CER {base_test[1]:.2f}%  words correct {base_test[2]:.2f}%")
print("(Round-2 model should show about 3.6% val CER. Much higher = you started from an older model.)")

## 11. Train
After every epoch: prints losses, CER, word accuracy and gradient norms; saves the best model; rewrites
every txt file. Stops at `EPOCHS`, at `PATIENCE`, or when `TIME_LIMIT_HOURS` would be passed.

In [ ]:
import csv
opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=1)
LAYERS = {f"conv{i+1}": [model.convs[i].weight, model.bns[i].weight, model.bns[i].bias] for i in range(5)}
LAYERS["lstm"] = list(model.lstm.parameters())
LAYERS["output"] = [model.out.weight]
LAYER_EVERY = 20               # per-layer gradient norms measured every 20 batches (saves time)

history, start_epoch = [], 1
best_cer, best_epoch = base_val[1], 0
if RESUME_FILE:
    ck = torch.load(RESUME_FILE, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"])
    if "scheduler" in ck: sched.load_state_dict(ck["scheduler"])
    history, best_cer, best_epoch = ck.get("history", []), ck["best_cer"], ck.get("best_epoch", 0)
    base_val = tuple(ck.get("base_val", base_val)); start_epoch = ck["epoch"] + 1
    best_src = os.path.join(os.path.dirname(RESUME_FILE), "best_model.pt")
    if os.path.exists(best_src):
        import shutil; shutil.copy(best_src, f"{OUT}/best_model.pt")
        tmp = HTR(len(CHARS) + 1); tmp.load_state_dict(torch.load(best_src, map_location="cpu")); save_npz(tmp, f"{OUT}/htr_weights_best.npz")
    print(f"Continuing after epoch {ck['epoch']} (best val CER so far {best_cer:.2f}% at epoch {best_epoch})")
if not os.path.exists(f"{OUT}/best_model.pt"):
    save_npz(model, f"{OUT}/htr_weights_best.npz"); torch.save(model.state_dict(), f"{OUT}/best_model.pt")

def write_all():
    """Rewrite every per-epoch txt file from the history (epoch 0 = start model)."""
    def put(name, header, lines):
        with open(f"{OUT}/{name}", "w") as f:
            f.write(header + "\n" + "".join(l + "\n" for l in lines))
    H = history
    put("train_loss.txt", "epoch\ttrain_loss", [f"{h['epoch']}\t{h['train_loss']:.5f}" for h in H])
    put("val_loss.txt", "epoch\tval_loss", [f"0\t{base_val[0]:.5f}"] + [f"{h['epoch']}\t{h['val_loss']:.5f}" for h in H])
    put("val_cer.txt", "epoch\tval_cer_percent", [f"0\t{base_val[1]:.3f}"] + [f"{h['epoch']}\t{h['val_cer']:.3f}" for h in H])
    put("val_word_accuracy.txt", "epoch\tval_word_accuracy_percent",
        [f"0\t{base_val[2]:.3f}"] + [f"{h['epoch']}\t{h['val_word_accuracy']:.3f}" for h in H])
    put("grad_norm.txt", "epoch\tmean_grad_norm\tmax_grad_norm",
        [f"{h['epoch']}\t{h['grad_norm_mean']:.5f}\t{h['grad_norm_max']:.5f}" for h in H])
    put("grad_norm_per_layer.txt", "epoch\t" + "\t".join(LAYERS),
        [f"{h['epoch']}\t" + "\t".join(f"{h['grad_' + k]:.6f}" for k in LAYERS) for h in H])
    put("learning_rate.txt", "epoch\tlearning_rate", [f"{h['epoch']}\t{h['lr']:.2e}" for h in H])
    put("epoch_time.txt", "epoch\tseconds", [f"{h['epoch']}\t{h['seconds']:.1f}" for h in H])
    if H:
        with open(f"{OUT}/history.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(H[0])); w.writeheader(); w.writerows(H)
write_all()

last_epoch_secs = 0
for epoch in range(start_epoch, EPOCHS + 1):
    used_h = (time.time() - RUN_START) / 3600
    if epoch > start_epoch and used_h + 1.15 * last_epoch_secs / 3600 > TIME_LIMIT_HOURS:
        print(f"Time limit: {used_h:.1f} h used, next epoch would pass {TIME_LIMIT_HOURS} h - stopping here.")
        break
    t0 = time.time(); model.train()
    loss_sum = n = 0
    grad_norms, layer_sums, layer_n = [], {k: 0.0 for k in LAYERS}, 0
    for step, (x, targets, lengths, texts) in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", mininterval=30)):
        logits = model(x.to(DEVICE, non_blocking=True))
        logp = F.log_softmax(logits, 2).permute(1, 0, 2)
        loss = ctc_loss(logp, targets, torch.full((len(texts),), T_STEPS, dtype=torch.long), lengths)
        opt.zero_grad()
        loss.backward()
        if step % LAYER_EVERY == 0:
            for k, params in LAYERS.items():
                layer_sums[k] += torch.norm(torch.stack([p.grad.norm() for p in params if p.grad is not None])).item()
            layer_n += 1
        grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), 5.0))
        opt.step()
        loss_sum += loss.detach() * len(texts); n += len(texts)
    train_loss = (loss_sum / n).item()
    grad_norms = torch.stack(grad_norms).float().cpu().numpy()
    val_loss, val_cer, val_acc, _ = evaluate(model, val_loader)
    lr = opt.param_groups[0]["lr"]
    sched.step(val_cer)
    secs = last_epoch_secs = time.time() - t0
    g_mean, g_max = float(np.mean(grad_norms)), float(np.max(grad_norms))
    per_layer = {k: v / max(1, layer_n) for k, v in layer_sums.items()}

    improved = val_cer < best_cer
    if improved:
        best_cer, best_epoch = val_cer, epoch
        save_npz(model, f"{OUT}/htr_weights_best.npz")
        torch.save(model.state_dict(), f"{OUT}/best_model.pt")
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_cer": val_cer,
                    "val_word_accuracy": val_acc, "grad_norm_mean": g_mean, "grad_norm_max": g_max, "lr": lr,
                    "seconds": secs, **{f"grad_{k}": v for k, v in per_layer.items()}})
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": opt.state_dict(),
                "scheduler": sched.state_dict(), "best_cer": best_cer, "best_epoch": best_epoch,
                "history": history, "base_val": list(base_val[:3])}, f"{OUT}/last_checkpoint.pt")
    write_all()

    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val CER {val_cer:.2f}% "
          f"| words correct {val_acc:.2f}% | grad norm mean {g_mean:.3f} max {g_max:.3f} | lr {lr:.1e} "
          f"| {secs/60:.1f} min{'  <- best, saved' if improved else ''}", flush=True)
    print("          gradient norm per layer: " + "  ".join(f"{k} {v:.4f}" for k, v in per_layer.items()), flush=True)

    if epoch - best_epoch >= PATIENCE:
        print(f"No improvement for {PATIENCE} epochs: stopping.")
        break

print(f"\nBest validation CER {best_cer:.2f}% at epoch {best_epoch} (start model: {base_val[1]:.2f}%)")

## 12. Graphs for the report

In [ ]:
H = pd.DataFrame(history)
def plot(cols, labels, ylabel, fname, title, base=None):
    if H.empty: return
    plt.figure(figsize=(7, 4.2))
    for c, l in zip(cols, labels):
        plt.plot(H["epoch"], H[c], marker="o", label=l)
    if base is not None:
        plt.axhline(base, ls="--", c="gray", label="start model")
    if best_epoch:
        plt.axvline(best_epoch, ls=":", c="green", label=f"best epoch ({best_epoch})")
    plt.xlabel("epoch"); plt.ylabel(ylabel); plt.title(title); plt.grid(alpha=.3); plt.legend()
    plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=150); plt.show()

plot(["train_loss", "val_loss"], ["training loss", "validation loss"], "CTC loss", "loss_curve.png",
     "Full dataset: training and validation loss")
plot(["val_cer"], ["validation CER"], "character error rate (%)", "cer_curve.png",
     "Full dataset: validation character error rate", base_val[1])
plot(["val_word_accuracy"], ["validation word accuracy"], "words fully correct (%)", "word_accuracy_curve.png",
     "Full dataset: validation word accuracy", base_val[2])
plot(["grad_norm_mean", "grad_norm_max"], ["mean", "max"], "gradient norm", "grad_norm_curve.png",
     "Full dataset: gradient norm per epoch (before clipping at 5)")
gcols = [f"grad_{k}" for k in LAYERS]
plot(gcols, list(LAYERS), "mean gradient norm", "grad_norm_per_layer.png", "Full dataset: gradient norm per layer")
plot(["lr"], ["learning rate"], "learning rate", "learning_rate.png", "Full dataset: learning rate")

## 13. Final test (writers never used for training)

In [ ]:
model.load_state_dict(torch.load(f"{OUT}/best_model.pt", map_location=DEVICE))
after_test = evaluate(model, test_loader, keep_examples=40)
row = lambda name, r: f"{name:28}{r[1]:10.2f}{r[2]:18.2f}{r[0]:12.3f}"
lines = ["Test set: SSL-HWD-Words writers never used for training or validation (same as rounds 1 and 2)",
         f"{'model':28}{'CER %':>10}{'words correct %':>18}{'CTC loss':>12}"]
if ORIGINAL_WEIGHTS:
    lines.append(row("original app model (IAM)", evaluate(new_model(ORIGINAL_WEIGHTS), test_loader)))
lines += [row("start model", base_test), row("after full-dataset training", after_test), "",
          f"Training words: {len(train_names):,} from {len(train_writers)} writers",
          f"Epochs run: {len(history)}; best epoch {best_epoch}; validation CER {base_val[1]:.2f}% -> {best_cer:.2f}%"]
text = "\n".join(lines); print(text)
open(f"{OUT}/test_results.txt", "w").write(text + "\n")
with open(f"{OUT}/sample_predictions.txt", "w") as f:
    f.write("correct text | start model | after full training\n")
    for (t, p0), (_, p1) in zip(base_test[3], after_test[3]):
        f.write(f"{t} | {p0} | {p1}\n")

## 14. One file to download

In [ ]:
import shutil
os.chdir("/kaggle/working")
shutil.make_archive("/kaggle/working/results_full", "zip", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"{f:28} {os.path.getsize(f'{OUT}/{f}') / 1e6:8.2f} MB")
from IPython.display import FileLink, display
display(FileLink("results_full.zip"))